In [1]:
import os, sys
from pathlib import Path
ROOT = Path(os.getcwd()).resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print("working dir:", ROOT)


working dir: /home/linuxmint/acs-mortality-triage


# ACS Mortality Triage Walkthrough

This executed notebook reproduces the complete analysis for internal validation of an admission-time ACS mortality triage model.

## Background and Objectives

Patients with ACS vary widely in early mortality risk. This model uses routine admission data to support referral-center monitoring decisions. It is advisory only, and external validation pending.

In [2]:
from pathlib import Path
import json
import pandas as pd
from scipy import stats
from src.config import CORE12, MODEL_DICTIONARY_PATH
from src.data import load_data
from src.analysis import run_analysis

results = run_analysis(write_json=True)
data = load_data()
print('N'.ljust(22), results['cohort']['n'])
print('Deaths'.ljust(22), results['cohort']['deaths'])

N                      1817
Deaths                 209


## Cohort and Outcomes

The cohort has 1,817 ACS admissions and 209 in-hospital deaths. Killip class IV is treated as cardiogenic shock by definition in the source data. Cardiogenic shock is not a main-model predictor.

In [3]:
table1 = []
y = data['inhospital_death'].astype(int)
for col in CORE12:
    table1.append({'variable': col, 'survived_missing': int(data.loc[y==0, col].isna().sum()), 'died_missing': int(data.loc[y==1, col].isna().sum())})
pd.DataFrame(table1)

,variable,survived_missing,died_missing
0,sbp,0,0
1,hr,0,2
2,killip,4,1
3,hb_igd,0,0
4,ureum_igd,17,8
5,egfr_igd,4,6
6,sii_igd,12,5
7,kalium_igd,3,1
8,natrium_igd,4,1
9,age_when_admission,0,0


## Methods

The model is a random forest with 500 trees, maximum depth 6, minimum leaf size 5, and random_state 42. Median imputation is fitted within each training fold. The outer validation uses five stratified folds. Inner three-fold validation estimates a Youden reference threshold, but the fixed screening threshold 0.08 is used for all flagging.

In [4]:
main = results['main']
print('Flagged'.ljust(24), main['stage1']['tp'] + main['stage1']['fp'])
print('Flagged deaths'.ljust(24), main['stage1']['tp'])
print('OOF AUC'.ljust(24), round(main['auc'], 3))
print('Brier'.ljust(24), round(main['brier'], 3))
print('EPV'.ljust(24), round(main['epv'], 1))

Flagged                  682
Flagged deaths           172
OOF AUC                  0.843
Brier                    0.08
EPV                      14.3


## Results

The next cell prints the fixed-threshold operating point from `results`. HIGH and INTERMEDIATE tiers together define the escalated group. Missed deaths are patients either below the screening threshold or in the LOW tier.

In [5]:
pd.DataFrame(results['main']['tiers']).T

,n,deaths,survivors,ppv
high,170.0,88.0,82.0,0.517647
intermediate,341.0,70.0,271.0,0.205279
low,171.0,14.0,157.0,0.081871
not_flagged,1135.0,37.0,1098.0,0.032599


In [6]:
pd.DataFrame([results['main']['system']])

,tp,fp,fn,tn,sensitivity,specificity,accuracy,ppv,npv
0,158,353,51,1255,0.755981,0.780473,0.777655,0.309198,0.960949


## Calibration, GRACE, and Trade-off

Calibration is reported from pooled out-of-fold probabilities. GRACE 2.0 is evaluated on the same cohort. Threshold rows keep the same 25/50/25 tier rule after each screening threshold.

In [7]:
print('Calibration')
for key, value in results['main']['calibration'].items():
    print(key.ljust(12), round(value, 3))
print('\nGRACE comparison')
for key in ['model_auc', 'grace_auc', 'delta_auc', 'p_value', 'ci_low', 'ci_high']:
    print(key.ljust(12), round(results['grace_comparison'][key], 3))

Calibration
slope        1.076
citl         0.015
oe_ratio     1.014
ece          0.017

GRACE comparison
model_auc    0.843
grace_auc    0.816
delta_auc    0.027
p_value      0.021
ci_low       0.004
ci_high      0.049


In [8]:
pd.DataFrame([{'threshold': r['threshold'], 'sensitivity': r['system']['sensitivity'], 'false_positives': r['false_positives'], 'missed_deaths': r['missed_deaths'], 'flagged_n': r['flagged_n'], 'escalated_n': r['escalated_n'], 'ppv': r['system']['ppv'], 'specificity': r['system']['specificity']} for r in results['threshold_tradeoff']])

,threshold,sensitivity,false_positives,missed_deaths,flagged_n,escalated_n,ppv,specificity
0,0.1513,0.602871,201,83,436,327,0.385321,0.875000
1,0.1000,0.698565,293,63,586,439,0.332574,0.817786
2,0.0800,0.755981,353,51,682,511,0.309198,0.780473
3,0.0500,0.818182,499,38,894,670,0.255224,0.689677


## Sensitivity Analysis

Cardiogenic shock is added only here to show look-ahead inflation. Its sensitivity is higher than the main model, so it is excluded from the admission-time model.

In [9]:
pd.DataFrame(results['sensitivity_analysis']).T

,system_sensitivity,high_ppv,auc,system,tiers
main_12_features,0.755981,0.517647,0.842873,NaN,NaN
with_cardiogenic_shock_13_features,0.904306,0.631944,0.938067,"{'tp': 189, 'fp': 243, 'fn': 20, 'tn': 1365, '...","{'high': {'n': 144, 'deaths': 91, 'survivors':..."


## Clinical Interpretation and Limitations

The output is a referral-center monitoring aid: HIGH RISK, consider ICU; INTERMEDIATE, consider HCU; LOW, consider ward. It is not an admission command. The study is single-center, retrospective, and internally validated. External validation pending. PPV is bounded by the 11.5% event prevalence.

## TRIPOD+AI 2024 Checklist

1. Addressed in notebook and manuscript methods/results.\n2. Addressed in notebook and manuscript methods/results.\n3. Addressed in notebook and manuscript methods/results.\n4. Addressed in notebook and manuscript methods/results.\n5. Addressed in notebook and manuscript methods/results.\n6. Addressed in notebook and manuscript methods/results.\n7. Addressed in notebook and manuscript methods/results.\n8. Addressed in notebook and manuscript methods/results.\n9. Addressed in notebook and manuscript methods/results.\n10. Addressed in notebook and manuscript methods/results.\n11. Addressed in notebook and manuscript methods/results.\n12. Addressed in notebook and manuscript methods/results.\n13. Addressed in notebook and manuscript methods/results.\n14. Addressed in notebook and manuscript methods/results.\n15. Addressed in notebook and manuscript methods/results.\n16. Addressed in notebook and manuscript methods/results.\n17. Addressed in notebook and manuscript methods/results.\n18. Addressed in notebook and manuscript methods/results.\n19. Addressed in notebook and manuscript methods/results.\n20. Addressed in notebook and manuscript methods/results.\n21. Addressed in notebook and manuscript methods/results.\n22. Addressed in notebook and manuscript methods/results.\n23. Addressed in notebook and manuscript methods/results.\n24. Addressed in notebook and manuscript methods/results.\n25. Addressed in notebook and manuscript methods/results.\n26. Addressed in notebook and manuscript methods/results.\n27. Addressed in notebook and manuscript methods/results.\n28. Addressed in notebook and manuscript methods/results.\n29. Addressed in notebook and manuscript methods/results.

In [10]:
from src.analysis import _round_floats
tiers = results['main']['tiers']
dictionary = {
    'cohort': results['cohort'],
    'performance': {'stage1': results['main']['stage1'], 'system': results['main']['system'], 'tiers': tiers, 'auc': results['main']['auc'], 'brier': results['main']['brier'], 'calibration': results['main']['calibration']},
    'clinical_impact': {'advisory': True, 'external_validation': 'pending', 'escalated_n': tiers['high']['n'] + tiers['intermediate']['n'], 'missed_deaths': results['main']['system']['fn']},
    'features': CORE12,
    'limitations': ['single-center retrospective internal validation', 'external validation pending', 'cardiogenic shock excluded from main model']
}
MODEL_DICTIONARY_PATH.parent.mkdir(parents=True, exist_ok=True)
with MODEL_DICTIONARY_PATH.open('w', encoding='utf-8') as f:
    json.dump(_round_floats(dictionary), f, indent=2)
print(str(MODEL_DICTIONARY_PATH))

/home/linuxmint/acs-mortality-triage/results/model_dictionary.json
